# Dataset preparation for training notebooks

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

CANDIDATE_FOLDERS = [Path.cwd(), Path('F:/isaaclab/prototype-thesis'),
                     Path('/mnt/f/isaaclab/prototype-thesis')]
PROJECT_DIR = next((p.resolve() for p in CANDIDATE_FOLDERS
                    if (p / 'scripts/build_driver_state_dataset.py').is_file() and (p / 'dms').is_dir()), None)
assert PROJECT_DIR is not None, 'Repository not found. Edit CANDIDATE_FOLDERS to locate scripts/ and dms/.'
os.chdir(PROJECT_DIR)
THESIS_DIR = next((p for p in [Path('F:/isaaclab/thesis'), Path('/mnt/f/isaaclab/thesis'),
                              PROJECT_DIR.parent / 'thesis', PROJECT_DIR]
                   if (p / 'driver_dataset.zip').is_file() or (p / 'YawDD.rar.gz').is_file()),
                  PROJECT_DIR.parent / 'thesis')
PHONE_ROOT = Path('datasets/ready_grouped').resolve()
GROUPS_CSV = PHONE_ROOT / 'split_manifest.csv'
PHONE_YAML = PHONE_ROOT / 'data.yaml'
STATE_ROOT = Path('datasets/driver_state').resolve()
STATE_YAML = STATE_ROOT / 'data.yaml'
NEED_PHONE = not STATE_YAML.is_file() and not (PHONE_YAML.is_file() and GROUPS_CSV.is_file())
print('Repository:', PROJECT_DIR)
print('Dataset archive folder:', THESIS_DIR)
print('Need phone preparation:', NEED_PHONE)


## Environment and reusable commands


In [ ]:
import csv
import json
import math
import random
import shutil
import time
from datetime import datetime
from collections import Counter, defaultdict
import collections

START = time.monotonic()

def log(message, level='INFO'):
    print(f'[{datetime.now():%H:%M:%S}] [+{time.monotonic()-START:,.0f}s] [{level}] {message}', flush=True)

def section(message):
    print('\n' + '=' * 76, flush=True)
    log(message, 'STEP')
    print('=' * 76, flush=True)

def progress(done, total, message):
    log(f'{message}: {done:,}/{total:,} ({100*done/total:.1f}%)' if total else message)

def run_verbose(cmd, *, cwd=PROJECT_DIR, allow_failure=False):
    cmd = [str(item) for item in cmd]
    section('Running ' + ' '.join(cmd))
    started = time.monotonic()
    with subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as p:
        for line in p.stdout:
            print(line, end='', flush=True)
        status = p.wait()
    log(f'Exit {status}; elapsed {(time.monotonic()-started)/60:.1f} minutes',
        'SUCCESS' if status == 0 else 'ERROR')
    if status and not allow_failure:
        raise RuntimeError(f'Command failed with exit code {status}. See the streamed output above.')
    return status

# speed-oriented starting point. Raise only if validation results warrant it.
MAX_TRAIN_IMAGES = 12000  # 0 means use all train images
EPOCHS = 35
BATCH = 16           # if CUDA runs out of memory, change to 8
PATIENCE = 8
IMGSZ = 320
WORKERS = 2          # conservative for Windows notebook + video processing
SEED = 0

log('pipeline configured')

In [ ]:
%pip install -r requirements.txt


In [ ]:
section('Checking the Python environment')
try:
    import cv2
    import torch
    import yaml
    import ultralytics
except ImportError as exc:
    raise RuntimeError('Install this repository\'s requirements into the notebook interpreter using '
                       f'\"{sys.executable}\" -m pip install -r requirements.txt') from exc
log(f'OpenCV {cv2.__version__}; Torch {torch.__version__}; Ultralytics {ultralytics.__version__}')
log('CUDA available: ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    log('CUDA GPU: ' + torch.cuda.get_device_name(0))
for asset in ['models/yolo11n.pt', 'models/face_landmarker.task']:
    if not Path(asset).is_file():
        log(f'Missing {asset}; downloading pretrained assets')
        run_verbose([sys.executable, 'scripts/download_models.py'])
        break
assert all(Path(f).is_file() for f in ['models/yolo11n.pt','models/face_landmarker.task'])
log('Pretrained models ready', 'SUCCESS')

## Prepare and group the public source dataset, if necessary


In [ ]:
if NEED_PHONE:
    section('Check files')
    from zipfile import ZipFile
    import gzip

    dataset_zip = THESIS_DIR / 'driver_dataset.zip'

    yawdd_archive = THESIS_DIR / 'YawDD.rar.gz'

    # Check both required files exist
    for file in (dataset_zip, yawdd_archive):
        assert file.is_file(), f"Missing Google Drive file: {file}"
        print(f"FOUND: {file.name} ({file.stat().st_size / 1e9:.2f} GB)")

    # Validate the YOLO dataset ZIP
    with ZipFile(dataset_zip) as archive:
        bad_file = archive.testzip()
        assert bad_file is None, f"Corrupt ZIP member: {bad_file}"

    print("Driver dataset ZIP is valid.")

    # Verify that the YawDD file has a readable GZIP header
    with gzip.open(yawdd_archive, "rb") as archive:
        header = archive.read(8)

    assert header.startswith((b"Rar!\x1a\x07\x00", b"Rar!\x1a\x07\x01")), (
        "The GZIP contents do not appear to be a RAR archive."
    )

    print("YawDD GZIP header and RAR signature are valid.")
    print("Both dataset archives are ready.")
else:
    print('Grouped/driver-state dataset found; source archive checks skipped.')


In [ ]:
if NEED_PHONE:
    section('Extract zip')
    log("Cell __ started.")
    from zipfile import ZipFile

    section("Extracting driver dataset")

    zip_path = THESIS_DIR / 'driver_dataset.zip'

    extract_path = Path("datasets/ready")

    assert zip_path.is_file(), (
        f"Dataset ZIP not found: {zip_path}"
    )

    log(f"Archive: {zip_path}")
    log(f"Archive size: {zip_path.stat().st_size / 1e9:.2f} GB")
    log(f"Destination: {extract_path.resolve()}")

    extract_path.mkdir(parents=True, exist_ok=True)

    started = time.monotonic()

    with ZipFile(zip_path) as archive:
        members = archive.infolist()
        total = len(members)

        log(f"Archive contains {total:,} entries.")

        for i, member in enumerate(members, start=1):
            archive.extract(member, extract_path)

            if i % 250 == 0 or i == total:
                progress(i, total, "Extracting archive")

    log(
        f"Extraction completed in "
        f"{time.monotonic() - started:,.1f}s",
        "SUCCESS",
    )

    yaml_files = list(extract_path.rglob("data.yaml"))

    log(f"Found {len(yaml_files)} data.yaml file(s).")

    for file in yaml_files:
        log(f"Dataset configuration: {file}")

    assert yaml_files, "No data.yaml found after extraction."
else:
    print('Source ZIP extraction skipped.')


In [ ]:
if NEED_PHONE:
    section('Check dataset before training')
    log("Cell __ started.")

    matches = list(Path("datasets/ready").rglob("data.yaml"))
    assert len(matches) == 1, f"Expected one data.yaml, found {len(matches)}"

    root = matches[0].parent.resolve()
    config = yaml.safe_load(matches[0].read_text())

    folders = {}
    for split, candidates in {
        "train": ["train"],
        "val": ["valid", "val"],
        "test": ["test"]
    }.items():
        found = next(
            (name for name in candidates if (root / name / "images").is_dir()),
            None
        )
        assert found, f"Missing {split} images folder"
        assert (root / found / "labels").is_dir(), f"Missing {split} labels"
        folders[split] = found

    config["path"] = str(root)
    for split, folder in folders.items():
        config[split] = f"{folder}/images"

    data_file = root / "data.yaml"
    data_file.write_text(yaml.safe_dump(config, sort_keys=False))

    print("DATASET READY")
    print("YAML:", data_file)
    print("Class names:", config["names"])
    for split, folder in folders.items():
        print(split, len(list((root / folder / "images").iterdir())), "files")
else:
    print('Source data.yaml preparation skipped.')


In [ ]:
if NEED_PHONE:
    section('Convert to bounding boxes')
    log("Cell __ started.")

    # Uses root, folders, and config from the preceding dataset check.
    assert "root" in globals()
    assert "folders" in globals()
    assert "config" in globals()

    n_classes = len(config["names"])

    label_files = sorted(
        p
        for folder in folders.values()
        for p in (root / folder / "labels").rglob("*.txt")
    )

    replacements = {}
    problems = []
    converted_polygons = 0

    section("Converting polygon annotations to bounding boxes")

    log(f"Label files found: {len(label_files):,}")
    started = time.monotonic()

    for file_index, label_file in enumerate(
        label_files, start=1
    ):
        original = label_file.read_text(encoding="utf-8")
        new_lines = []
        changed = False

        if file_index % 500 == 0:
            progress(
                file_index,
                len(label_files),
                "Reading label files",
            )
            log(
                f"Polygons converted so far: "
                f"{converted_polygons:,}"
            )

        for line_number, line in enumerate(
            original.splitlines(), start=1
        ):
            parts = line.split()

            if not parts:
                continue

            # Already a YOLO detection bounding box.
            if len(parts) == 5:
                new_lines.append(line)
                continue

            # A polygon needs a class ID and at least
            # three pairs of X/Y coordinates.
            if len(parts) < 7 or len(parts) % 2 != 1:
                problems.append(
                    f"{label_file}:{line_number}: "
                    f"unexpected {len(parts)} values"
                )
                continue

            try:
                class_id = int(parts[0])
                coordinates = [float(v) for v in parts[1:]]

                if not 0 <= class_id < n_classes:
                    raise ValueError("Invalid class ID")

                if not all(
                    math.isfinite(v) and 0 <= v <= 1
                    for v in coordinates
                ):
                    raise ValueError(
                        "Polygon coordinates must be between 0 and 1"
                    )

                xs = coordinates[0::2]
                ys = coordinates[1::2]

                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)

                width = x_max - x_min
                height = y_max - y_min

                if width <= 0 or height <= 0:
                    raise ValueError("Zero-area polygon")

                center_x = (x_min + x_max) / 2
                center_y = (y_min + y_max) / 2

                new_lines.append(
                    f"{class_id} "
                    f"{center_x:.8f} "
                    f"{center_y:.8f} "
                    f"{width:.8f} "
                    f"{height:.8f}"
                )

                converted_polygons += 1
                changed = True

            except (ValueError, OverflowError) as error:
                problems.append(
                    f"{label_file}:{line_number}: {error}"
                )

        if changed:
            replacements[label_file] = (
                "\n".join(new_lines) + "\n"
            )

    # Avoid partially converting the dataset if
    # any polygon cannot be interpreted safely.
    if problems:
        print("Examples of problematic annotations:")
        for problem in problems[:20]:
            print(problem)

        raise RuntimeError(
            f"{len(problems)} annotations need manual review. "
            "No label files have been changed."
        )

    # Modify only the extracted dataset labels.
    log(f"Writing {len(replacements):,} converted label files.")
    for file_index, (label_file, contents) in enumerate(replacements.items(), 1):
        if file_index % 500 == 0 or file_index == len(replacements):
            progress(file_index, len(replacements), "Saving converted labels")
        label_file.write_text(contents, encoding="utf-8")

    log("CONVERSION COMPLETED", "SUCCESS")

    log(f"Polygon annotations converted: {converted_polygons:,}")
    log(f"Label files updated: {len(replacements):,}")
    log(f"Total label files examined: {len(label_files):,}")

    log(
        f"Total elapsed time: "
        f"{time.monotonic() - started:,.1f}s"
    )
else:
    print('Polygon conversion skipped.')


## Regroup source images and check split leakage


In [ ]:
if NEED_PHONE:
    section('Group similar images into splits')
    log("Cell 17 started.")
    from collections import defaultdict, Counter
    import yaml

    from dms.data.audit import BANDS, dhash, read_image
    from dms.data.splits import original_name

    # Start with the dataset whose polygons were converted
    source_root = Path("datasets/ready").resolve()
    source_folders = dict(folders)

    # Save the reorganized dataset separately
    new_root = Path("datasets/ready_grouped").resolve()

    assert not new_root.exists(), (
        f"{new_root} already exists. "
        "Inspect or remove that temporary output before rerunning."
    )

    # STEP 1 - Collect images and label paths
    records = []

    log("Scanning source image and label directories.")
    for split, folder in source_folders.items():
        log(f"Scanning {split} in {folder}")
        image_folder = source_root / folder / "images"
        label_folder = source_root / folder / "labels"

        for image in sorted(image_folder.iterdir()):
            if image.suffix.lower() not in {
                ".jpg", ".jpeg", ".png", ".bmp", ".webp"
            }:
                continue

            records.append({
                "split": split,
                "image": image,
                "label": label_folder / (image.stem + ".txt"),
            })

    N = len(records)
    log(f"Found {N:,} images; building source and similarity groups.")
    assert N >= 3, "Dataset has too few images."

    # STEP 2 - Prepare similarity groups
    parent = list(range(N))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # Group Roboflow variants of the same original image
    first_source = {}

    for i, record in enumerate(records):
        if (i+1) % 5000 == 0: progress(i+1, N, "Grouping original filenames")
        source_name = original_name(record["image"].name)

        if source_name in first_source:
            union(i, first_source[source_name])
        else:
            first_source[source_name] = i

    # STEP 3 - Group near-identical images
    # Uses the same dHash and four-bit threshold as
    # scripts/audit_dataset.py

    buckets = defaultdict(list)
    hashes = []

    log("Calculating image perceptual hashes and connecting similar images. This may take time.")
    for i, record in enumerate(records):
        gray, _ = read_image(record["image"])

        assert gray is not None, (
            f"Unreadable image: {record['image']}"
        )

        h = dhash(gray)
        candidates = set()

        for band, (lo, hi) in enumerate(BANDS):
            key = (
                band,
                (h >> lo) & ((1 << (hi - lo)) - 1)
            )

            candidates.update(buckets[key])
            buckets[key].append(i)

        for j in candidates:
            if (h ^ hashes[j]).bit_count() <= 4:
                union(i, j)

        hashes.append(h)

        if (i + 1) % 500 == 0 or i + 1 == N:
            progress(i + 1, N, "Analyzing image similarity")

    # STEP 4 - Collect groups
    components = defaultdict(list)

    for i in range(N):
        components[find(i)].append(i)

    print("Images:", N)
    print("Similarity groups:", len(components))
    print(
        "Largest group:",
        max(map(len, components.values()))
    )

    assert len(components) >= 3, (
        "Too few separate similarity groups for 3 splits. "
        "Obtain independently sourced validation/test data."
    )

    # STEP 5 - Assign whole groups to splits
    target = {
        "train": 0.70 * N,
        "val": 0.15 * N,
        "test": 0.15 * N,
    }

    counts = Counter()
    assignment = {}

    for group, members in sorted(
        components.items(),
        key=lambda item: -len(item[1])
    ):
        split = min(
            ("train", "val", "test"),
            key=lambda s: counts[s] / target[s]
        )

        assignment[group] = split
        counts[split] += len(members)

    print("New split sizes:", dict(counts))

    assert counts["val"] >= 100 and counts["test"] >= 100, (
        "Validation or test has fewer than 100 images. "
        "The similarity groups are too concentrated. "
        "Additional independent data is needed."
    )

    # STEP 6 - Create the new dataset
    folder_name = {
        "train": "train",
        "val": "valid",
        "test": "test",
    }

    manifest = []

    log("Writing grouped images, labels, and manifest.")
    written = 0
    for group, members in components.items():
        split = assignment[group]

        for i in members:
            record = records[i]

            # Prefix keeps filenames unique after regrouping
            name = (
                record["split"] + "_"
                + record["image"].name
            )

            destination = new_root / folder_name[split]

            image_out = (
                destination / "images" / name
            )

            label_out = (
                destination / "labels"
                / (Path(name).stem + ".txt")
            )

            image_out.parent.mkdir(
                parents=True, exist_ok=True
            )

            label_out.parent.mkdir(
                parents=True, exist_ok=True
            )

            # Try hard links to avoid duplicating image bytes
            try:
                os.link(record["image"], image_out)
            except OSError:
                shutil.copy2(record["image"], image_out)

            if record["label"].is_file():
                shutil.copy2(record["label"], label_out)

            written += 1
            if written % 500 == 0 or written == N:
                progress(written, N, "Writing grouped dataset")
            manifest.append({
                "file": name,
                "group": f"cluster_{group}",
                "split": split,
            })

    # STEP 7 - Save group assignments
    manifest_path = new_root / "split_manifest.csv"

    with manifest_path.open(
        "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=["file", "group", "split"]
        )

        writer.writeheader()
        writer.writerows(manifest)

    # STEP 8 - Write the new data.yaml
    new_config = dict(config)
    new_config["path"] = str(new_root)

    for split, folder in folder_name.items():
        new_config[split] = folder + "/images"

    new_data_file = new_root / "data.yaml"

    new_data_file.write_text(
        yaml.safe_dump(new_config, sort_keys=False)
    )

    # STEP 9 - Run the full repository audit
    log("Starting full dataset audit, including cross-split similarity.")

    audit_command = [
        sys.executable,
        "scripts/audit_dataset.py",
        "--data", str(new_root),
        "--groups", str(manifest_path),
    ]

    log("Executing: " + " ".join(audit_command))

    result = subprocess.run(
        audit_command,
        capture_output=True,
        text=True,
        errors="replace",
        check=False,
    )

    print("\n" + "=" * 70)
    print("DATASET AUDIT OUTPUT")
    print("=" * 70)

    print(result.stdout or "(No standard output)")

    if result.stderr:
        print("\nAUDIT STDERR")
        print(result.stderr)

    print("\nAudit exit code:", result.returncode)

    # STEP 10 - Read the detailed audit report
    report_path = (
        Path("results/dataset_audit")
        / new_root.name
        / "audit.json"
    )

    if report_path.is_file():
        report = json.loads(
            report_path.read_text(encoding="utf-8")
        )

        print("\n" + "=" * 70)
        print("REPORTED DATASET PROBLEMS")
        print("=" * 70)

        for problem in report.get("problems", []):
            print("-", problem)

        print("\nSPLIT INFORMATION")

        for split, details in report.get(
            "splits", {}
        ).items():
            print(
                f"{split}: "
                f"{details.get('images', '?')} images, "
                f"{details.get('label_error_files', '?')} "
                "label-error files"
            )

    else:
        print(
            "No audit.json found at:",
            report_path
        )

    # STEP 11 - Stop if the audit found problems
    if result.returncode != 0:
        raise RuntimeError(
            "Dataset audit failed. "
            "Read the detailed findings printed above."
        )

    # STEP 12 - Use the new dataset in later cells
    root = new_root
    config = new_config
    data_file = new_data_file
    folders = folder_name

    log("Grouped dataset passed the repository audit.", "SUCCESS")

    print("Grouped dataset passed the repository audit.")
    print("New dataset:", root)
else:
    print('Existing grouped dataset reused.')


In [ ]:
if not STATE_YAML.is_file():
    section('Checking grouped dataset')

    PHONE_ROOT = Path('datasets/ready_grouped').resolve()
    GROUPS_CSV = PHONE_ROOT / 'split_manifest.csv'
    PHONE_YAML = PHONE_ROOT / 'data.yaml'

    assert PHONE_ROOT.is_dir() and PHONE_YAML.is_file() and GROUPS_CSV.is_file(), (
        'Grouped data not ready. First complete the extract / polygon conversion / grouping '
        'steps above; do not delete an existing grouped folder.')
    with GROUPS_CSV.open(newline='', encoding='utf-8-sig') as fh:
        reader = csv.DictReader(fh)
        assert {'file','group','split'} <= set(reader.fieldnames or []), 'Incorrect group manifest columns'
        source_rows = list(reader)

    print('Images in manifest:', len(source_rows))
    print('Original split counts:', dict(Counter(x['split'] for x in source_rows)))

    cluster_counts = Counter(x['group'] for x in source_rows)
    print('Similarity clusters:',len(cluster_counts),'largest:',cluster_counts.most_common(1))
    assert len(source_rows)>0 and len(cluster_counts)>2

    log('Grouped dataset and manifest found. Reusing existing work.', 'SUCCESS')
else:
    print('Four-class data exists; grouped source checks skipped.')


In [ ]:
if not STATE_YAML.is_file():
    section('Audit source dataset')
    SOURCE_AUDIT = Path('results/quick_driver_state/source_audit').resolve()

    status = run_verbose([sys.executable,'scripts/audit_dataset.py','--data',PHONE_ROOT,
                          '--groups',GROUPS_CSV,'--out',SOURCE_AUDIT])
    source_report = json.loads((SOURCE_AUDIT/'audit.json').read_text(encoding='utf-8'))

    print('Reported problems:',source_report.get('problems', []))
    assert status == 0, 'Source group audit found leakage'

    # The script may return zero for non-leakage problems. Inspect and stop for annotation errors.
    for item in source_report.get('problems',[]):
        if 'label' in str(item).lower() or 'corrupt' in str(item).lower():
            raise RuntimeError('Source audit reports a label/image problem needing review: ' + str(item))

    log('Source audit completed. Check report before interpreting scores.', 'SUCCESS')
else:
    print('Four-class data exists; grouped source audit skipped.')


## Prepare YawDD and generate four-class training data, if needed


In [ ]:
YAWDD_DIR = Path('data/YawDD').resolve()
if not STATE_YAML.is_file():
    section('Locating or extracting YawDD videos')
    RAR_GZIP = THESIS_DIR / 'YawDD.rar.gz'
    VIDEO_EXTS = {'.avi','.mp4','.mkv','.mov'}
    def available_videos():
        return sorted(p for p in YAWDD_DIR.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS)

    videos = available_videos() if YAWDD_DIR.is_dir() else []
    if videos:
        log(f'Already extracted: {len(videos):,} video files, skipping archive extraction')
    else:
        assert RAR_GZIP.is_file(), f'YawDD archive missing: {RAR_GZIP}'
        YAWDD_DIR.mkdir(parents=True, exist_ok=True)
        rar = Path('data/YawDD_intermediate.rar').resolve()
        if not rar.is_file():
            log(f'Decompressing {RAR_GZIP.name} ({RAR_GZIP.stat().st_size/1e9:.2f} GB)')
            with gzip.open(RAR_GZIP,'rb') as source, rar.open('wb') as dest:
                copied=0
                while True:
                    block=source.read(64*1024*1024)
                    if not block:
                        break
                    dest.write(block)
                    copied+=len(block)
                    if copied%(512*1024*1024)<64*1024*1024:
                        log(f'GZIP -> RAR: {copied/(1024**3):.2f} GiB written')
        else:
            log('Reusing existing intermediate .rar file')
        candidates=[shutil.which('7z'),shutil.which('7za'),Path('C:/Program Files/7-Zip/7z.exe'),
                    Path('C:/Program Files (x86)/7-Zip/7z.exe')]
        sevenzip=next((str(x) for x in candidates if x and Path(x).is_file()),None)
        if not sevenzip:
            raise RuntimeError('Install 7-Zip and ensure 7z.exe is available.')
        run_verbose([sevenzip,'x','-y',f'-o{YAWDD_DIR}',rar])
        videos=available_videos()
        assert videos,'No videos found after extracting YawDD.'
        rar.unlink(missing_ok=True)
    log(f'YawDD ready: {len(videos)} video files', 'SUCCESS')
    print('Samples:',[str(p) for p in videos[:3]])
else:
    print('Four-class dataset exists; YawDD extraction skipped.')


In [ ]:
section('Build driver-state dataset, or reuse completed output')
STATE_ROOT=Path('datasets/driver_state').resolve()
STATE_YAML=STATE_ROOT/'data.yaml'
if STATE_YAML.is_file() and any((STATE_ROOT/'images/train').glob('*')):
    log(f'Existing four-class dataset found at {STATE_ROOT}; skipping expensive MediaPipe extraction')
else:
    if STATE_ROOT.exists() and any(STATE_ROOT.iterdir()):
        raise RuntimeError(f'{STATE_ROOT} exists but is incomplete. Inspect it before rerunning the builder.')
    run_verbose([sys.executable,'scripts/build_driver_state_dataset.py',
                 '--phone-dataset',PHONE_ROOT,'--phone-groups',GROUPS_CSV,
                 '--yawdd',YAWDD_DIR,'--out',STATE_ROOT,
                 '--exclude-dash','--scan-stride','3','--every-sec','2.0',
                 '--event-every-sec','0.4','--seed',str(SEED)])
assert STATE_YAML.is_file(), 'Four-class data.yaml missing'
raw_names = yaml.safe_load(
    STATE_YAML.read_text(encoding='utf8')
)['names']

if isinstance(raw_names, dict):
    state_names = [
        raw_names.get(i, raw_names.get(str(i)))
        for i in range(len(raw_names))
    ]
else:
    state_names = list(raw_names)

print('Classes:', state_names)

assert state_names == [
    'eye_closed',
    'eye_open',
    'phone',
    'yawn'
], 'Incorrect four-class names/order'

log('Four-class data.yaml ready', 'SUCCESS')

## Audit the four-class data without silently modifying held-out images


In [ ]:
STATE_MANIFEST = STATE_ROOT / 'split_manifest.csv'
STATE_AUDIT = Path('results/quick_driver_state/four_class_audit').resolve()
assert STATE_MANIFEST.is_file(), f'Missing four-class split manifest: {STATE_MANIFEST}'
run_verbose([sys.executable, 'scripts/audit_dataset.py', '--data', STATE_ROOT,
             '--groups', STATE_MANIFEST, '--out', STATE_AUDIT])
report = json.loads((STATE_AUDIT / 'audit.json').read_text(encoding='utf-8'))
print('Reported problems:', report.get('problems', []))
assert not report.get('problems'), 'Investigate dataset audit problems before using held-out results.'


## Prepare the smaller YOLO training split and ESP32-look images


In [ ]:
section('Prepare training subset without touching held-out splits')

FAST_ROOT=Path(f'datasets/driver_state_fast_{MAX_TRAIN_IMAGES or "all"}').resolve()
FAST_YAML=FAST_ROOT/'data.yaml'
source_yaml=yaml.safe_load(STATE_YAML.read_text(encoding='utf8'))
source_train=STATE_ROOT/'images/train'

assert source_train.is_dir()
train_imgs=sorted(p for p in source_train.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})

def source_label(img):
    return STATE_ROOT/'labels/train'/f'{img.stem}.txt'

def classes_of(img):
    label=source_label(img)
    if not label.is_file():
        return set()
    found=set()
    for line in label.read_text(encoding='utf8').splitlines():
        parts=line.split()
        if parts:
            found.add(int(parts[0]))
    return found

if FAST_YAML.is_file():
    log('Existing fast subset found; keeping it unchanged')
else:
    if FAST_ROOT.exists() and any(FAST_ROOT.iterdir()):
        raise RuntimeError(f'{FAST_ROOT} is partially populated. Inspect before rerunning.')
    rng=random.Random(SEED)
    target=min(len(train_imgs),MAX_TRAIN_IMAGES) if MAX_TRAIN_IMAGES>0 else len(train_imgs)
    
    # Gather rare classes first. The rest is uniformly sampled with a fixed seed.
    pool=list(train_imgs)
    rng.shuffle(pool)
    chosen=set()
    priorities=(0,3,2)  # closed eye, yawn, phone; extra examples where possible
    per_class=min(1500,max(1,target//6))
    for class_id in priorities:
        count=0
        for img in pool:
            if len(chosen)>=target or count>=per_class:
                break
            if img not in chosen and class_id in classes_of(img):
                chosen.add(img)
                count+=1
        log(f'Priority class {state_names[class_id]}: {count} examples chosen')
    for img in pool:
        if len(chosen)>=target:
            break
        chosen.add(img)
    (FAST_ROOT/'images/train').mkdir(parents=True,exist_ok=True)
    (FAST_ROOT/'labels/train').mkdir(parents=True,exist_ok=True)
    for idx,img in enumerate(sorted(chosen),1):
        dest=FAST_ROOT/'images/train'/img.name
        try:
            os.link(img,dest)
        except OSError:
            shutil.copy2(img,dest)
        label=source_label(img)
        destlabel=FAST_ROOT/'labels/train'/label.name
        if label.is_file():
            shutil.copy2(label,destlabel)
        else:
            destlabel.write_text('',encoding='utf8')
        if idx%1000==0 or idx==len(chosen):
            progress(idx,len(chosen),'Linking training subset')
    
    # Absolute held-out paths let the repository's loader retain ALL val/test images.
    fast_config={'path':str(FAST_ROOT),'train':'images/train',
                 'val':str((STATE_ROOT/'images/val').resolve()),
                 'test':str((STATE_ROOT/'images/test').resolve()),
                 'names':state_names}
    FAST_YAML.write_text(yaml.safe_dump(fast_config,sort_keys=False),encoding='utf8')

print('Original train images:',len(train_imgs))
print('Fast train images:',sum(1 for p in (FAST_ROOT/'images/train').iterdir() if p.is_file()))
log('Training subset ready', 'SUCCESS')

In [ ]:
section('Generate or reuse ESP32-look training dataset')

ESP_ROOT=Path(f'datasets/driver_state_esp32_fast_{MAX_TRAIN_IMAGES or "all"}').resolve()
ESP_YAML=ESP_ROOT/'data.yaml'
ESP_TEST_YAML=ESP_ROOT/'data_esp32_test.yaml'

if ESP_YAML.is_file() and ESP_TEST_YAML.is_file():
    log('ESP32-look dataset already exists; skipping regeneration')
else:
    if ESP_ROOT.exists() and any(ESP_ROOT.iterdir()):
        raise RuntimeError(f'{ESP_ROOT} incomplete; inspect it before regenerating.')
    run_verbose([sys.executable,'scripts/make_esp32_look.py',
                 '--data',FAST_YAML,'--out',ESP_ROOT,
                 '--copies','1','--replace','--seed',str(SEED)])
assert ESP_YAML.is_file() and ESP_TEST_YAML.is_file()

log('ESP32-look dataset ready', 'SUCCESS')

## Build fusion features from the shared dataset

In [ ]:
DATA_YAML = STATE_YAML
DATA_MANIFEST = STATE_MANIFEST
FUSION_TABLE = PROJECT_DIR / 'datasets/fusion/esca_phase2_train_val.csv'


In [ ]:
section('Build fusion features with separate train/validation/test splits')

REBUILD_FUSION_FEATURES = False

if REBUILD_FUSION_FEATURES or not FUSION_TABLE.is_file():
    cmd = [sys.executable, 'scripts/build_fusion_dataset.py',
            '--data', DATA_YAML, '--out', FUSION_TABLE,
            '--splits', 'train', 'val', 'test']
    if DATA_MANIFEST.is_file():
        cmd += ['--groups', DATA_MANIFEST]
    run_verbose(cmd)
else:
    log(f'Reusing fusion table: {FUSION_TABLE}')
exists_or_explain(FUSION_TABLE, 'fusion feature CSV')
with FUSION_TABLE.open(newline='', encoding='utf-8') as stream:
    reader = csv.DictReader(stream)
    targets = collections.defaultdict(lambda: collections.Counter())
    for row in reader:
        split = row['split']
        assert split in ('train', 'val', 'test'), (
            f'Unexpected fusion dataset split: {split}')
        for head in ('eye_closed', 'yawn'):
            if row.get(head) in ('0', '1'):
                targets[(split, head)][row[head]] += 1
for split in ('train','val','test'):
    for head in ('eye_closed','yawn'):
        print(f'{split}/{head}: negative={targets[(split,head)]["0"]}, positive={targets[(split,head)]["1"]}')
for head in ('eye_closed','yawn'):
    assert targets[('train',head)]['0'] >= 20 and targets[('train',head)]['1'] >= 20, (
        f'Not enough TRAIN examples for {head}. Inspect data labels and fusion extraction summary.')
    assert sum(targets[('val',head)].values()) > 0, f'No validation examples for {head}.'

Now you can run either `training-1.ipynb` (four-class YOLO detector) or `training-2.ipynb` (two-head fusion model)